# Bquant Connector

Bloomberg BQL 데이터를 조회하여 아시아 주식 데이터를 DataFrame으로 반환하는 파이프라인입니다.

**BQuant 환경에서 직접 실행 가능합니다.**

## 1. Setup & Imports

In [ ]:
from __future__ import annotations

import logging
import time
from typing import Any, Iterable, Optional, Sequence

import bql
import pandas as pd

# BQL Service 초기화 (BQuant 환경에서 자동으로 연결됩니다)
svc = bql.Service()
print("BQL Service 연결 완료")

## 2. Configuration

조회할 Bloomberg 필드와 출력 컬럼 이름을 정의합니다.

In [ ]:
logger = logging.getLogger("bquant_connector")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

# Bloomberg 필드 -> 출력 컬럼명 매핑
FIELD_SPECS: list[tuple[str, str]] = [
    # Identifiers
    ("TICKER", "Ticker"),
    ("NAME", "NAME"),
    ("ID_ISIN", "ID_ISIN"),
    ("ID_BB_GLOBAL", "ID_BB_GLOBAL"),
    # Classifications
    ("GICS_INDUSTRY_NAME", "GICS_INDUSTRY_NAME"),
    ("MARKET_STATUS", "MARKET_STATUS"),
    ("PRIMARY_EXCH_NAME", "PRIMARY_EXCHANGE"),
    ("STK_MKT_CONNECT_ELIGIBILITY", "STOCK_CONNECT"),
    # Fundamentals
    ("RETURN_ON_INVESTED_CAPITAL", "ROIC (%)"),
    ("FREE_CASH_FLOW_YIELD", "FCF Yield (%)"),
    ("BS_TOT_ASSET", "Tot. Assets"),
    ("EBITDA", "EBITDA"),
    ("SALES_REV_TURN", "Revenue"),
    ("OPER_INC", "Op. Income"),
    ("BOOK_VAL_PER_SH", "BV/Share"),
    # Price
    ("PX_LAST", "Last Price"),
]

NUMERIC_COLUMNS: tuple[str, ...] = (
    "ROIC (%)",
    "FCF Yield (%)",
    "Tot. Assets",
    "EBITDA",
    "Revenue",
    "Op. Income",
    "BV/Share",
    "Last Price",
)

OUTPUT_COLUMNS: list[str] = [spec[1] for spec in FIELD_SPECS]
print(f"조회 필드 수: {len(FIELD_SPECS)}")
print(f"출력 컬럼: {OUTPUT_COLUMNS}")

## 3. Core Functions

In [ ]:
def get_universe(
    service: bql.Service,
    tickers: Optional[Sequence[str]] = None,
    eqs_screen_name: Optional[str] = None,
) -> Any:
    """BQL universe 생성: ticker 리스트 또는 저장된 EQS 스크린 이름으로 생성합니다."""
    if (tickers is not None) == (eqs_screen_name is not None):
        raise ValueError("tickers 또는 eqs_screen_name 중 하나만 입력하세요.")

    # BQuant 환경에서는 service 인스턴스에서 univ에 접근합니다
    univ = getattr(service, "univ", None) or getattr(bql, "univ", None)
    if univ is None:
        raise RuntimeError(
            "universe 헬퍼를 찾을 수 없습니다. BQuant/bql 버전을 확인하세요."
        )

    if tickers is not None:
        cleaned = [t.strip() for t in tickers if str(t).strip()]
        if not cleaned:
            raise ValueError("tickers는 비어있지 않은 문자열 시퀀스여야 합니다.")
        return univ.list(cleaned)

    assert eqs_screen_name is not None
    name = eqs_screen_name.strip()
    for attr in ("EqsScreen", "eqsscreen", "eqs", "EQS"):
        ctor = getattr(univ, attr, None)
        if callable(ctor):
            return ctor(name)

    raise RuntimeError(
        "EQS universe를 사용할 수 없습니다. tickers= 파라미터를 사용하거나 BQuant/bql 버전을 확인하세요."
    )


def build_bql_request(
    service: bql.Service,
    universe: Any,
    field_specs: Sequence[tuple[str, str]] = FIELD_SPECS,
) -> Any:
    """Bloomberg 필드에 대한 BQL get 요청을 생성합니다."""
    items = tuple(spec[0] for spec in field_specs)
    return service.get(*items).for_(universe)


def _response_to_dataframe(response: Any) -> pd.DataFrame:
    """BQL 응답을 DataFrame으로 변환합니다. 다양한 BQuant 버전에 대응합니다."""
    if response is None:
        return pd.DataFrame()

    for attr in ("dataframe", "to_dataframe", "as_dataframe"):
        fn = getattr(response, attr, None)
        if callable(fn):
            df = fn()
            if isinstance(df, pd.DataFrame):
                return df

    composite = getattr(bql, "composite", None)
    if composite is not None:
        for attr in ("to_dataframe", "dataframe"):
            fn = getattr(composite, attr, None)
            if callable(fn):
                try:
                    df = fn(response)
                    if isinstance(df, pd.DataFrame):
                        return df
                except Exception:
                    pass

    if hasattr(response, "data") and callable(response.data):
        try:
            payload = response.data()
            if isinstance(payload, pd.DataFrame):
                return payload
        except Exception:
            pass

    frames: list[pd.DataFrame] = []
    try:
        for part in response:
            if isinstance(part, pd.DataFrame):
                frames.append(part)
            else:
                for attr in ("dataframe", "to_dataframe", "df"):
                    fn = getattr(part, attr, None)
                    if callable(fn):
                        dfp = fn()
                        if isinstance(dfp, pd.DataFrame):
                            frames.append(dfp)
                            break
    except TypeError:
        pass

    if frames:
        if len(frames) == 1:
            return frames[0]
        return pd.concat(frames, axis=1)

    raise TypeError(
        "BQL 응답을 DataFrame으로 변환할 수 없습니다. execute 반환 타입을 확인하세요."
    )


def fetch_bql_data(
    service: bql.Service,
    request: Any,
    *,
    max_retries: int = 7,
    base_delay_sec: float = 1.0,
    max_delay_sec: float = 25.0,
) -> Any:
    """BQL 요청을 실행합니다. 일시적 오류 시 지수 백오프로 재시도합니다."""
    last_error: Optional[BaseException] = None
    for attempt in range(max_retries):
        try:
            return service.execute(request)
        except Exception as exc:
            last_error = exc
            msg = str(exc).lower()
            retryable = any(
                token in msg
                for token in (
                    "timeout", "timed out", "throttl", "rate",
                    "503", "502", "429", "temporar", "try again", "overloaded",
                )
            )
            if attempt >= max_retries - 1 or not retryable:
                logger.exception(
                    "BQL 실행 실패 (시도 %s회): %s", attempt + 1, exc,
                )
                raise

            delay = min(max_delay_sec, base_delay_sec * (2 ** attempt))
            logger.warning(
                "BQL 실행 시도 %s/%s 실패 (%s); %.1f초 후 재시도",
                attempt + 1, max_retries, exc, delay,
            )
            time.sleep(delay)

    assert last_error is not None
    raise last_error


def clean_data(
    df: pd.DataFrame,
    field_specs: Sequence[tuple[str, str]] = FIELD_SPECS,
    numeric_columns: Iterable[str] = NUMERIC_COLUMNS,
) -> pd.DataFrame:
    """BQL 컬럼 이름을 정리하고, MultiIndex를 평탄화하고, 데이터 타입을 변환합니다."""
    if df.empty:
        return pd.DataFrame().reindex(columns=pd.Index(OUTPUT_COLUMNS))

    out = df.copy()

    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            "_".join(str(level) for level in col if str(level) not in ("", "nan"))
            for col in out.columns.values
        ]

    direct = {spec[0]: spec[1] for spec in field_specs}
    rename_map: dict[Any, str] = {}
    for col in out.columns:
        key = str(col)
        if key in direct:
            rename_map[col] = direct[key]
            continue
        for bql_item, header in field_specs:
            if key == bql_item or key.endswith(f"_{bql_item}") or key.startswith(
                f"{bql_item}_"
            ):
                rename_map[col] = header
                break

    out = out.rename(columns=rename_map)

    for header in OUTPUT_COLUMNS:
        if header not in out.columns:
            out[header] = pd.NA

    out = out.loc[:, OUTPUT_COLUMNS]

    for col in numeric_columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    for col in ("Ticker", "NAME", "ID_ISIN", "ID_BB_GLOBAL"):
        out[col] = out[col].astype("string")

    for col in ("GICS_INDUSTRY_NAME", "MARKET_STATUS", "PRIMARY_EXCHANGE", "STOCK_CONNECT"):
        out[col] = out[col].astype("string")

    return out


def run_pipeline(
    tickers: Optional[Sequence[str]] = None,
    eqs_screen_name: Optional[str] = None,
    service: Optional[bql.Service] = None,
    field_specs: Sequence[tuple[str, str]] = FIELD_SPECS,
) -> pd.DataFrame:
    """전체 파이프라인 실행: universe -> request -> execute -> DataFrame"""
    svc = service or bql.Service()
    universe = get_universe(svc, tickers=tickers, eqs_screen_name=eqs_screen_name)
    request = build_bql_request(svc, universe, field_specs=field_specs)
    response = fetch_bql_data(svc, request)
    raw = _response_to_dataframe(response)
    return clean_data(raw, field_specs=field_specs)

print("함수 정의 완료")

## 4. 데이터 조회 실행

아시아 주요 종목 9개를 조회합니다. `tickers` 리스트를 수정하여 원하는 종목을 조회할 수 있습니다.

In [ ]:
# 조회할 종목 리스트 (원하는 종목으로 수정 가능)
tickers = [
    "700 HK Equity",    # Tencent
    "9988 HK Equity",   # Alibaba
    "3690 HK Equity",   # Meituan
    "005930 KS Equity", # Samsung Electronics
    "6758 JP Equity",   # Sony
    "7203 JP Equity",   # Toyota
    "2330 TT Equity",   # TSMC
    "D05 SP Equity",    # DBS Group
    "9984 JP Equity",   # SoftBank
]

print(f"조회 종목 수: {len(tickers)}")
for t in tickers:
    print(f"  - {t}")

In [ ]:
# 파이프라인 실행
df = run_pipeline(tickers=tickers, service=svc)
print(f"조회 완료: {len(df)}개 종목")

## 5. 결과 확인

In [ ]:
# 전체 데이터 출력
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
df

In [ ]:
# 기본 통계
df.describe()

In [ ]:
# 데이터 타입 확인
df.dtypes

## 6. 커스텀 조회 예시

아래 셀의 주석을 해제하여 다른 방식으로 데이터를 조회할 수 있습니다.

In [ ]:
# 예시 1: 다른 종목 조회
# custom_tickers = ["AAPL US Equity", "MSFT US Equity", "GOOGL US Equity"]
# df_custom = run_pipeline(tickers=custom_tickers, service=svc)
# df_custom

In [ ]:
# 예시 2: 저장된 EQS 스크린으로 조회
# df_eqs = run_pipeline(eqs_screen_name="My Saved Screen", service=svc)
# df_eqs

In [ ]:
# 예시 3: 모듈로 import하여 사용 (bquant_connector.py가 같은 디렉토리에 있을 때)
# from bquant_connector import run_pipeline as rp
# df_import = rp(tickers=["700 HK Equity"], service=svc)
# df_import